# Centroid refinement of the harmonic fork, scored with the competition metric

The fork writes integer voxel centres (a z voxel is 1.625 um). Each variant moves every
node to the intensity-weighted centre of mass of a smoothed window around it, iterated
like mean-shift, then scores the result with and without rounding back to integers.


In [ ]:
import subprocess, sys
from pathlib import Path

# The Kaggle image lacks zarr; install it offline from the fork's support-pack wheels.
try:
    import zarr  # noqa: F401
except ImportError:
    wheels = [w for w in Path("/kaggle/input").rglob("*.whl")
              if w.name.lower().split("-")[0] in {"zarr", "numcodecs", "donfig", "crc32c", "google_crc32c"}]
    print("installing", sorted(w.name for w in wheels))
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *map(str, wheels)], check=True)


In [ ]:
import functools
import time
from pathlib import Path

import numpy as np
import pandas as pd
import zarr
from scipy.ndimage import gaussian_filter

COMP_DIR = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
if not COMP_DIR.exists():
    COMP_DIR = Path("/kaggle/input/biohub-cell-tracking-during-development")
TEST_DIR = COMP_DIR / "test"
_subs = [p for p in Path("/kaggle/input").rglob("submission.csv") if "harmonic" in str(p)]
assert _subs, "harmonic fork submission.csv not attached"
SUB = pd.read_csv(_subs[0])
print(_subs[0], SUB.shape)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import zarr
from scipy.optimize import linear_sum_assignment

CMP_SCALE = np.array([1.625, 0.40625, 0.40625])
CMP_MAX_D_UM = 7.0
CMP_TRAIN = COMP_DIR / "train"


def cmp_load_gt(stem: str) -> dict:
    root = CMP_TRAIN / f"{stem}.geff"
    g = zarr.open(str(root), mode="r")
    ids = np.asarray(g["nodes/ids"])
    row_of = {int(v): i for i, v in enumerate(ids)}
    raw = np.asarray(g["edges/ids"]).reshape(-1, 2)
    with (root / "zarr.json").open() as f:
        meta = json.load(f)["attributes"]["geff"]
    return {
        "t": np.asarray(g["nodes/props/t/values"]).astype(np.int64),
        "zyx": np.stack([np.asarray(g[f"nodes/props/{k}/values"]) for k in "zyx"], 1).astype(float),
        "edges": np.array([[row_of[int(a)], row_of[int(b)]] for a, b in raw], dtype=np.int64).reshape(-1, 2),
        "est_total": int((meta.get("extra") or {}).get("estimated_number_of_nodes", 0)),
    }


def cmp_match(p_t, p_um, g_t, g_um):
    matched = np.full(len(p_t), -1, np.int64)
    for t in np.intersect1d(np.unique(p_t), np.unique(g_t)):
        pi, gi = np.flatnonzero(p_t == t), np.flatnonzero(g_t == t)
        d = np.sqrt(((g_um[gi][:, None] - p_um[pi][None]) ** 2).sum(2))
        w = np.where(d <= CMP_MAX_D_UM, 1.0 / (1.0 + d), -1.0)
        if not (w > -1.0).any():
            continue
        r, c = linear_sum_assignment(w, maximize=True)
        keep = w[r, c] > -1.0
        matched[pi[c[keep]]] = gi[r[keep]]
    return matched


def cmp_score(sub: pd.DataFrame, stem: str) -> dict:
    """Node-count-adjusted edge Jaccard for one dataset of a submission."""
    gt = cmp_load_gt(stem)
    part = sub[sub["dataset"] == stem]
    nodes = part[part["row_type"] == "node"]
    row_of = {int(v): i for i, v in enumerate(nodes["node_id"])}
    p_t = nodes["t"].to_numpy(np.int64)
    p_zyx = nodes[["z", "y", "x"]].to_numpy(float)
    ed = part[part["row_type"] == "edge"]
    e = np.array([(row_of[int(a)], row_of[int(b)]) for a, b in zip(ed["source_id"], ed["target_id"])],
                 dtype=np.int64).reshape(-1, 2)
    n_est = gt["est_total"]
    if not len(e):
        return {"J": 0.0, "adj": 0.0, "n_pred": len(nodes), "n_est": n_est}

    m = cmp_match(p_t, p_zyx * CMP_SCALE, gt["t"], gt["zyx"] * CMP_SCALE)
    e = np.unique(e, axis=0)
    e = e[p_t[e[:, 1]] - p_t[e[:, 0]] == 1]
    e = e[np.lexsort((e[:, 1], e[:, 0]))]
    rank = np.ones(len(e), np.int64)
    for i in range(1, len(e)):
        rank[i] = rank[i - 1] + 1 if e[i, 0] == e[i - 1, 0] else 1
    e = e[rank <= 2]

    ms, mt = m[e[:, 0]], m[e[:, 1]]
    gt_set = set(map(tuple, gt["edges"].tolist()))
    tp = sum((int(a), int(b)) in gt_set for a, b in zip(ms, mt) if a >= 0 and b >= 0)
    n_gt = len(gt["t"])
    gt_out = np.bincount(gt["edges"][:, 0], minlength=n_gt)
    gt_in = np.bincount(gt["edges"][:, 1], minlength=n_gt)
    graded = (np.where(ms >= 0, gt_out[np.clip(ms, 0, None)] > 0, False)
              | np.where(mt >= 0, gt_in[np.clip(mt, 0, None)] > 0, False))
    fp, fn = int(graded.sum()) - tp, len(gt["edges"]) - tp
    j = tp / max(tp + fp + fn, 1)
    adj = max(0.0, j * (1 - 0.1 * (len(nodes) - n_est) / n_est)) if n_est else j
    return {"J": round(j, 4), "adj": round(adj, 4), "tp": tp, "fp": fp, "fn": fn,
            "n_pred": len(nodes), "n_est": n_est, "node_ratio": round(len(nodes) / max(n_est, 1), 3)}




In [ ]:
cmp_load_gt = functools.lru_cache(maxsize=None)(cmp_load_gt)

SCALE = np.array([1.625, 0.40625, 0.40625])
VARIANTS = [
    {"name": f"s{s}_r{r}_i{i}", "smooth_um": s, "radius_um": r, "iters": i}
    for s in (0.5, 1.0) for r in (1.5, 2.5, 4.0) for i in (1, 3)
]


def window_offsets(radius_um):
    h = np.ceil(radius_um / SCALE).astype(int)
    g = np.stack(np.meshgrid(*[np.arange(-k, k + 1) for k in h], indexing="ij"), -1).reshape(-1, 3)
    return g[((g * SCALE) ** 2).sum(1) <= radius_um ** 2]


def refine_frame(vol, zyx, radius_um, iters):
    """Mean-shift each centre towards the background-subtracted intensity centre of mass."""
    off = window_offsets(radius_um)
    hi = np.array(vol.shape) - 1
    pos = zyx.astype(float).copy()
    for _ in range(iters):
        c = np.rint(pos).astype(int)
        idx = c[:, None, :] + off[None]
        inside = ((idx >= 0) & (idx <= hi)).all(2)
        idx = np.clip(idx, 0, hi)
        v = vol[idx[..., 0], idx[..., 1], idx[..., 2]]
        v = np.where(inside, v, np.nan)
        w = np.clip(v - np.nanmin(v, 1, keepdims=True), 0, None)
        w = np.nan_to_num(w)
        tot = w.sum(1)
        ok = tot > 0
        new = c + (w[:, :, None] * off[None]).sum(1) / np.where(ok, tot, 1)[:, None]
        pos = np.where(ok[:, None], new, pos)
    # Runaway guard: a node that drifted beyond the window radius keeps its original centre.
    drift = np.sqrt((((pos - zyx) * SCALE) ** 2).sum(1))
    pos[drift > radius_um] = zyx[drift > radius_um]
    return np.clip(pos, 0, hi)


refined = {v["name"]: SUB.copy() for v in VARIANTS}
for col in "zyx":
    for df in refined.values():
        df[col] = df[col].astype(float)

t0 = time.time()
nodes_all = SUB[SUB["row_type"] == "node"]
for stem, nodes in nodes_all.groupby("dataset"):
    img = zarr.open(str(TEST_DIR / f"{stem}.zarr"), mode="r")["0"]
    for t, fr in nodes.groupby("t"):
        raw = np.asarray(img[int(t)], dtype=np.float32)
        zyx = fr[["z", "y", "x"]].to_numpy(float)
        smoothed = {s: gaussian_filter(raw, s / SCALE) for s in {v["smooth_um"] for v in VARIANTS}}
        for v in VARIANTS:
            new = refine_frame(smoothed[v["smooth_um"]], zyx, v["radius_um"], v["iters"])
            refined[v["name"]].loc[fr.index, ["z", "y", "x"]] = new
    print(f"{stem}: {len(nodes)} nodes refined, {time.time() - t0:.0f}s")


In [ ]:
stems = sorted(s for s in SUB["dataset"].astype(str).unique() if (CMP_TRAIN / f"{s}.geff").exists())
node_mask = SUB["row_type"] == "node"


def shift_um(df):
    d = (df.loc[node_mask, ["z", "y", "x"]].to_numpy(float) - SUB.loc[node_mask, ["z", "y", "x"]].to_numpy(float)) * SCALE
    return float(np.sqrt((d ** 2).sum(1)).mean())


rows = []
cases = {"base": SUB}
for name, df in refined.items():
    cases[name] = df
    r = df.copy()
    r[["z", "y", "x"]] = np.rint(r[["z", "y", "x"]].to_numpy(float))
    cases[name + "_int"] = r
for name, df in cases.items():
    for stem in stems:
        rows.append({"variant": name, "dataset": stem, **cmp_score(df, stem)})
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/refine_results.csv", index=False)

summary = res.groupby("variant")[["tp", "fp", "fn"]].sum()
summary["J_micro"] = summary["tp"] / (summary["tp"] + summary["fp"] + summary["fn"])
summary["adj_mean"] = res.groupby("variant")["adj"].mean()
summary["mean_shift_um"] = [shift_um(cases[v]) for v in summary.index]
summary["dJ_vs_base"] = summary["J_micro"] - summary.loc["base", "J_micro"]
summary = summary.sort_values("J_micro", ascending=False)
summary.to_csv("/kaggle/working/refine_summary.csv")
print(summary.round(4).to_string())
print(res.pivot(index="variant", columns="dataset", values="J").round(4).to_string())


In [ ]:
# How far are the fork's matched centres from GT, and how many labelled cells go unmatched?
diag = []
nodes = SUB[node_mask]
for stem in stems:
    gt = cmp_load_gt(stem)
    part = nodes[nodes["dataset"] == stem]
    p_t = part["t"].to_numpy(np.int64)
    p_um = part[["z", "y", "x"]].to_numpy(float) * CMP_SCALE
    g_um = gt["zyx"] * CMP_SCALE
    m = cmp_match(p_t, p_um, gt["t"], g_um)
    hit = m >= 0
    err = p_um[hit] - g_um[m[hit]]
    d = np.sqrt((err ** 2).sum(1))
    diag.append({"dataset": stem, "gt_nodes": len(gt["t"]), "matched": int(hit.sum()),
                 "d_median": np.median(d), "d_p90": np.percentile(d, 90), "frac_gt_2um": (d > 2).mean(),
                 "abs_dz": np.abs(err[:, 0]).mean(), "abs_dy": np.abs(err[:, 1]).mean(),
                 "abs_dx": np.abs(err[:, 2]).mean()})
print(pd.DataFrame(diag).round(3).to_string())
